In [ ]:
# =====================================================================
# BLOCO 1: Configuração geral (instalação, secrets, clientes, funções)
# Rode este bloco UMA VEZ por sessão do Colab.
# =====================================================================
!pip install -q trafilatura google-genai notion-client

import os
import re
import json
import trafilatura
from datetime import datetime
from google import genai
from google.colab import userdata
from notion_client import Client as NotionClient

# --- Secrets -----------------------------------------------------------
try:
    CHAVE_GEMINI = userdata.get('GEMINI_API_KEY')
except Exception:
    CHAVE_GEMINI = None
    print("ERRO: Secret 'GEMINI_API_KEY' não encontrado.")

try:
    NOTION_DATABASE_ID = userdata.get('NOTION_DATABASE_ID_MTG_TRANSLATOR')
except Exception:
    NOTION_DATABASE_ID = None
    print("ERRO: Secret 'NOTION_DATABASE_ID_MTG_TRANSLATOR' não encontrado.")

gemini_client = genai.Client(api_key=CHAVE_GEMINI) if CHAVE_GEMINI else None
notion = NotionClient(auth=userdata.get("NOTION_TOKEN_MTG_TRANSLATOR"))

# --- Configurações fixas -------------------------------------------------
MODELO_ESCOLHIDO = 'gemini-2.5-flash'  # ou 'gemini-3.1-pro' para priorizar qualidade literária
STATUS_NAME = "Not started"            # coluna "Status" — sempre esse valor na publicação inicial

# --- Glossário de MTG (repositório GitHub dedicado, versionado) ---------
GITHUB_REPO_URL = "https://github.com/ThaisMosken/magic_story_translator.git"
GLOSSARY_FILE_PATH = "glossary_mtg.md"  # caminho do .md dentro do repositório — ajuste se mudar de lugar
FORCE_UPDATE_GLOSSARY = False             # True para puxar mudanças do repo a cada execução

REPO_LOCAL_FOLDER = "magic_story_translator"

if not os.path.exists(REPO_LOCAL_FOLDER):
    !git clone {GITHUB_REPO_URL}
elif FORCE_UPDATE_GLOSSARY:
    !git -C {REPO_LOCAL_FOLDER} pull

def load_glossary(md_path: str) -> dict:
    """Lê um glossário em Markdown no formato '- termo em inglês → tradução'
    e retorna um dicionário. Cabeçalhos (#, ##), linhas em branco e
    comentários HTML (<!-- ... -->) são ignorados."""
    vocabulary = {}
    in_comment = False
    with open(md_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("<!--"):
                in_comment = True
            if in_comment:
                if "-->" in line:
                    in_comment = False
                continue
            if line.startswith("#"):
                continue
            line = line.lstrip("-*").strip()
            if "→" in line:
                en, pt = line.split("→", 1)
                vocabulary[en.strip()] = pt.strip()
    return vocabulary

MTG_VOCABULARY = load_glossary(os.path.join(REPO_LOCAL_FOLDER, GLOSSARY_FILE_PATH))
print(f"✅ Glossário carregado com {len(MTG_VOCABULARY)} termos.")

# --- Extração do texto original (sem precisar de PDF/Drive) -------------
def extract_story(url: str) -> dict:
    downloaded = trafilatura.fetch_url(url)
    if downloaded is None:
        raise RuntimeError(f"Não foi possível baixar a página: {url}")

    text = trafilatura.extract(
        downloaded,
        include_comments=False,
        include_tables=False,
        include_images=True,
        favor_precision=True,
    )
    metadata = trafilatura.extract_metadata(downloaded)

    if not text or len(text) < 500:
        raise RuntimeError(
            "Texto extraído parece incompleto — verifique manualmente a página "
            "(pode ser que essa página em específico use outra estrutura)."
        )

    # trafilatura geralmente já retorna a data em formato ISO (YYYY-MM-DD),
    # mas caso venha em outro formato, tentamos normalizar aqui.
    raw_date = metadata.date if metadata else None
    release_date_iso = None
    if raw_date:
        try:
            release_date_iso = datetime.fromisoformat(raw_date).date().isoformat()
        except ValueError:
            release_date_iso = raw_date  # mantém como veio; ajuste manual se necessário

    return {
        "title": metadata.title if metadata else None,
        "author": metadata.author if metadata else None,
        "date": release_date_iso,
        "text": text,
    }

# --- Prompts de tradução (Gemini) ---------------------------------------
def build_translation_prompt(text: str, vocabulary: dict) -> str:
    vocab_lines = "\n".join(f"- {en} → {pt}" for en, pt in vocabulary.items())
    return f"""Você é um assistente especializado em tradução e formatação de conteúdos da web para Markdown (.md), destinado ao Notion. Seu objetivo é fornecer uma versão traduzida para o Português Brasileiro (PT-BR) que preserve integralmente o texto e a estrutura visual original.

O texto de origem abaixo já foi extraído automaticamente de uma página web (incluindo eventuais tags de imagem em Markdown, no formato ![alt](url), na posição em que aparecem no original).

Siga rigorosamente estas diretrizes:

1. TRADUÇÃO INTEGRAL (SEM RESUMOS):
- Traduza o texto completo de ponta a ponta.
- Nunca resuma, omita parágrafos, pule seções ou condense informações.
- Mantenha termos técnicos ou nomes próprios consagrados em inglês quando fizer sentido no contexto, preferencialmente em itálico.

2. DIRETRIZES DE MAGIC: THE GATHERING (MTG):
- A tradução deve obrigatoriamente seguir os termos oficiais do jogo utilizados nas cartas em português do Brasil (PT-BR): nomes de personagens (Planeswalkers, criaturas lendárias), locais (planos, continentes, faculdades, marcos geográficos), feitiços, raças e entidades.
- Glossário obrigatório (use exatamente estas traduções quando o termo aparecer):
{vocab_lines}

3. PRESERVAÇÃO DE FORMATAÇÃO MARKDOWN:
- Use cabeçalhos correspondentes à hierarquia original (#, ##, ###).
- Preserve rigorosamente negritos (**texto**) e itálicos (*texto*) nos mesmos locais do original.
- Mantenha listas com marcadores (- ou *) e listas numeradas exatamente onde aparecem.
- Formate blocos de citação com (>).
- NÃO inclua marcadores de sistema, colchetes de indexação, links de rastreamento ou indexadores automáticos (como "[cite: 1]") no texto final.

4. IMAGENS:
- Sempre que encontrar uma tag ![alt](url) no texto de origem, repita-a EXATAMENTE como está (mesma URL), na mesma posição — não invente, não altere e não remova a URL.
- Traga a legenda de crédito logo abaixo, traduzida e em itálico (ex: *Arte por: Nome do Artista*), caso exista no original.

5. ENTREGA:
- Devolva apenas o texto traduzido em Markdown, sem nenhum comentário, preâmbulo ou explicação (não escreva coisas como "aqui está a tradução").

Texto original:
{text}
"""

def build_title_prompt(title: str, vocabulary: dict) -> str:
    vocab_lines = "\n".join(f"- {en} → {pt}" for en, pt in vocabulary.items())
    return f"""Traduza o título abaixo, de uma matéria/conto de Magic: The Gathering, do inglês para o português brasileiro (PT-BR).

Regras:
- Use a terminologia oficial de Magic: The Gathering em português sempre que aplicável (nomes de planos, arcos de história, mecânicas).
- Glossário obrigatório (use exatamente estas traduções quando o termo aparecer):
{vocab_lines}
- Devolva APENAS o título traduzido, em uma única linha, sem aspas, sem comentários e sem explicações.

Título original:
{title}
"""

# --- Markdown -> blocos do Notion ---------------------------------------
IMAGE_RE = re.compile(r'^!\[(.*?)\]\((.*?)\)\s*$')
HEADING_RE = re.compile(r'^(#{1,3})\s+(.*)$')
BULLET_RE = re.compile(r'^[-*]\s+(.*)$')
NUMBERED_RE = re.compile(r'^\d+\.\s+(.*)$')

def parse_inline_markdown(text: str) -> list:
    """Converte **negrito** e *itálico* em rich_text do Notion.
    Cada segmento também respeita o limite de 2000 caracteres do Notion."""
    tokens = re.split(r'(\*\*.+?\*\*|\*.+?\*)', text)
    rich_text = []
    for token in tokens:
        if not token:
            continue
        if token.startswith('**') and token.endswith('**'):
            content, annotations = token[2:-2], {"bold": True}
        elif token.startswith('*') and token.endswith('*'):
            content, annotations = token[1:-1], {"italic": True}
        else:
            content, annotations = token, {}
        for i in range(0, len(content), 2000):
            piece = {"type": "text", "text": {"content": content[i:i + 2000]}}
            if annotations:
                piece["annotations"] = annotations
            rich_text.append(piece)
    return rich_text or [{"type": "text", "text": {"content": ""}}]

def chunk_plain_text(text: str, max_len: int = 2000) -> list:
    """Rede de segurança: garante que nenhum parágrafo isolado passe de 2000
    caracteres, mesmo depois do parsing de Markdown (quebra por frases)."""
    if len(text) <= max_len:
        return [text]
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for s in sentences:
        if len(current) + len(s) + 1 > max_len:
            if current:
                chunks.append(current)
            if len(s) > max_len:
                for i in range(0, len(s), max_len):
                    chunks.append(s[i:i + max_len])
                current = ""
            else:
                current = s
        else:
            current = f"{current} {s}" if current else s
    if current:
        chunks.append(current)
    return chunks

def markdown_to_notion_blocks(markdown_text: str) -> list:
    """Converte o Markdown traduzido em uma lista de blocos do Notion
    (heading_1/2/3, quote, bulleted/numbered list, image, paragraph)."""
    blocks = []
    lines = markdown_text.split('\n')
    paragraph_buffer = []

    def flush_paragraph():
        if not paragraph_buffer:
            return
        paragraph = ' '.join(paragraph_buffer).strip()
        paragraph_buffer.clear()
        if not paragraph:
            return
        for chunk in chunk_plain_text(paragraph):
            blocks.append({
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": parse_inline_markdown(chunk)},
            })

    i = 0
    while i < len(lines):
        line = lines[i].rstrip()
        stripped = line.strip()

        if not stripped:
            flush_paragraph()
            i += 1
            continue

        img_match = IMAGE_RE.match(stripped)
        if img_match:
            flush_paragraph()
            _, url = img_match.group(1), img_match.group(2)
            blocks.append({
                "object": "block",
                "type": "image",
                "image": {"type": "external", "external": {"url": url}},
            })
            if i + 1 < len(lines) and lines[i + 1].strip().startswith('*') and lines[i + 1].strip().endswith('*'):
                blocks.append({
                    "object": "block",
                    "type": "paragraph",
                    "paragraph": {"rich_text": parse_inline_markdown(lines[i + 1].strip())},
                })
                i += 1
            i += 1
            continue

        heading_match = HEADING_RE.match(stripped)
        if heading_match:
            flush_paragraph()
            heading_type = f"heading_{len(heading_match.group(1))}"
            blocks.append({
                "object": "block",
                "type": heading_type,
                heading_type: {"rich_text": parse_inline_markdown(heading_match.group(2))},
            })
            i += 1
            continue

        if stripped.startswith('>'):
            flush_paragraph()
            quote_lines = []
            while i < len(lines) and lines[i].strip().startswith('>'):
                quote_lines.append(lines[i].strip().lstrip('>').strip())
                i += 1
            blocks.append({
                "object": "block",
                "type": "quote",
                "quote": {"rich_text": parse_inline_markdown(' '.join(quote_lines))},
            })
            continue

        bullet_match = BULLET_RE.match(stripped)
        if bullet_match:
            flush_paragraph()
            blocks.append({
                "object": "block",
                "type": "bulleted_list_item",
                "bulleted_list_item": {"rich_text": parse_inline_markdown(bullet_match.group(1))},
            })
            i += 1
            continue

        numbered_match = NUMBERED_RE.match(stripped)
        if numbered_match:
            flush_paragraph()
            blocks.append({
                "object": "block",
                "type": "numbered_list_item",
                "numbered_list_item": {"rich_text": parse_inline_markdown(numbered_match.group(1))},
            })
            i += 1
            continue

        paragraph_buffer.append(stripped)
        i += 1

    flush_paragraph()
    return blocks

# --- Publicação no Notion -------------------------------------------------
def publish_story(
    database_id: str,
    story_meta: dict,
    translated_text: str,
    original_url: str,
    arc: str,
    tags: list,
    status_name: str,
    translated_title: str = None,
) -> str:
    all_blocks = markdown_to_notion_blocks(translated_text)

    # A API do Notion aceita no máximo 100 blocos filhos por requisição
    # (tanto na criação da página quanto em blocks.children.append)
    first_batch, remaining_batches = all_blocks[:100], all_blocks[100:]

    page_title = translated_title or story_meta["title"] or "Conto sem título"

    properties = {
        "Name": {"title": [{"text": {"content": page_title}}]},
        "Author": {"rich_text": [{"text": {"content": story_meta["author"] or ""}}]},
        "Arc": {"select": {"name": arc}},
        "Tags": {"multi_select": [{"name": t} for t in tags]},
        "Status": {"status": {"name": status_name}},
        "Original URL": {"url": original_url},
    }

    # "Release Date" só é enviado se conseguimos extrair/normalizar a data
    if story_meta.get("date"):
        properties["Release Date"] = {"date": {"start": story_meta["date"]}}

    page = notion.pages.create(
        parent={"database_id": database_id},
        properties=properties,
        children=first_batch,
    )

    for i in range(0, len(remaining_batches), 100):
        notion.blocks.children.append(
            block_id=page["id"],
            children=remaining_batches[i:i + 100],
        )

    return page["id"]

# --- Orquestrador: extrai, traduz e publica em uma única chamada --------
def translate_and_publish(story_url: str, arc_name: str, tags: list, dry_run: bool = False) -> None:
    if not dry_run and not gemini_client:
        print("❌ Não é possível continuar: CHAVE_GEMINI ausente.")
        return
    if not NOTION_DATABASE_ID:
        print("❌ Não é possível continuar: NOTION_DATABASE_ID ausente.")
        return

    print(f"🔎 Extraindo conteúdo de: {story_url}")
    try:
        story = extract_story(story_url)
    except Exception as e:
        print(f"❌ Erro na extração: {e}")
        return

    print(f"📖 Título original: {story['title']}")
    print(f"✍️  Autor: {story['author']}")
    print(f"📅 Data: {story['date']}")
    print(f"📝 Tamanho do texto extraído: {len(story['text'])} caracteres")

    if dry_run:
        print("🧪 DRY RUN ativo: pulando chamadas ao Gemini, usando texto de teste no lugar da tradução.")
        translated_text = (
            f"# {story['title'] or 'Conto de teste'}\n\n"
            "Este é um **parágrafo de teste** em negrito, e este trecho está em *itálico*.\n\n"
            "> Esta é uma citação de teste, só para validar a formatação.\n\n"
            "- Primeiro item de lista\n"
            "- Segundo item de lista\n\n"
            "Parágrafo final de teste, sem depender do Gemini."
        )
        translated_title = f"[TESTE] {story['title']}" if story.get("title") else "[TESTE] Conto sem título"
    else:
        print("🌐 Traduzindo corpo do texto...")
        response = gemini_client.models.generate_content(
            model=MODELO_ESCOLHIDO,
            contents=build_translation_prompt(story["text"], MTG_VOCABULARY),
        )
        translated_text = response.text

        translated_title = None
        if story.get("title"):
            print("🌐 Traduzindo título...")
            title_response = gemini_client.models.generate_content(
                model=MODELO_ESCOLHIDO,
                contents=build_title_prompt(story["title"], MTG_VOCABULARY),
            )
            translated_title = title_response.text.strip().strip('"').strip()
            print(f"📖 Título traduzido: {translated_title}")

    try:
        print("🚀 Publicando conto traduzido no Notion...")
        page_id = publish_story(
            NOTION_DATABASE_ID,
            story,
            translated_text,
            story_url,
            arc=arc_name,
            tags=tags,
            status_name=STATUS_NAME,
            translated_title=translated_title,
        )
        print(f"✅ Sucesso! Página criada: https://notion.so/{page_id.replace('-', '')}")
    except Exception as e:
        print(f"❌ Erro na integração com Notion: {e}")

print("✅ Ambiente configurado. Preencha o BLOCO 2 com os dados do conto e execute.")

In [ ]:
# =====================================================================
# BLOCO 2: Conto atual — preencha e rode este bloco para cada tradução
# =====================================================================
STORY_URL = "COLE_A_URL_AQUI"
ARC_NAME = "COLE_O_NOME_DO_ARCO_AQUI"       # precisa já existir como opção na coluna "Arc" (select)
TAGS = ["TAG_1", "TAG_2"]             # coluna "Tags" (multi-select)

translate_and_publish(STORY_URL, ARC_NAME, TAGS)

# Sem cota do Gemini disponível? Teste extração + publicação no Notion assim:
# translate_and_publish(STORY_URL, ARC_NAME, TAGS, dry_run=True)